<a href="https://colab.research.google.com/github/Wilson1994/DTA-2026/blob/main/ML/Classification_DecisionTreeClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Overfitting & underfitting. Cross-validation

In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Щоб результати були однаковими щоразу (відтворюваність)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Бібліотеки готові ✅")

Бібліотеки готові ✅


In [50]:
m = 1200
tenure   = np.random.randint(1, 72, m)           # місяців з нами
monthly  = np.random.normal(70, 25, m).clip(15, 150)  # щомісячна оплата, $
support  = np.random.poisson(1.5, m)             # звернень у підтримку за рік
age      = np.random.randint(18, 75, m)          # вік клієнта

# Прихована логіка ризику відтоку (модель її не знає):
risk = (
    -0.05 * tenure        # довше з нами → менший ризик
    + 0.02 * monthly      # дорожчий тариф → трохи більший ризик
    + 0.45 * support      # багато звернень у підтримку → більший ризик
    - 0.01 * age          # старші клієнти трохи лояльніші
    + np.random.normal(0, 0.7, m)
)
prob = 1 / (1 + np.exp(-(risk - 0.5)))   # перетворюємо ризик на ймовірність 0..1
churn = (np.random.rand(m) < prob).astype(int)

df = pd.DataFrame({
    "tenure": tenure, "monthly": monthly.round(1),
    "support": support, "age": age, "churn": churn,
})

print("Частка клієнтів, що пішли:", f"{df['churn'].mean():.1%}")
df.head()

Частка клієнтів, що пішли: 38.1%


,tenure,monthly,support,age,churn
0,51,54.7,0,25,0
1,18,92.8,2,59,0
2,58,87.0,1,26,0
3,44,35.1,1,73,0
4,16,52.4,3,67,0


In [51]:
from sklearn.model_selection import train_test_split

X = df[['tenure', 'monthly', 'support', 'age']]  # features - ознаки
X = df[['churn']]  # features - ознаки

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Навчальна вибірка:", X_train.shape[0], "клієнтів")
print("Тестова вибірка:", X_test.shape[0], "клієнтів")

ValueError: Found input variables with inconsistent numbers of samples: [1200, 12000000]

In [44]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

deep_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
deep_tree.fit(X_train, y_train)

acc_train = accuracy_score(y_train, deep_tree.predict(X_train))
acc_test = accuracy_score(y_test, deep_tree.predict(X_test))

print("Глубокое дерево без ограничений (переобучение)")
print(f"Train accuracy = {acc_train:.2%}")
print(f"Train accuracy = {acc_test:.2%}")
print(f"Разрыв = {(acc_train - acc_test):.2%} - ознака зубріння\n")

shallow = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
shallow.fit(X_train, y_train)

acc_train = accuracy_score(y_train, shallow.predict(X_train))
acc_test = accuracy_score(y_test, shallow.predict(X_test))

print("Простое дерево без ограничений (переобучение)")
print(f"Train accuracy = {acc_train:.2%}")
print(f"Train accuracy = {acc_test:.2%}")
print(f"Разрыв = {(acc_train - acc_test):.2%}")

Глубокое дерево без ограничений (переобучение)
Train accuracy = 63.43%
Train accuracy = 63.43%
Разрыв = -0.00% - ознака зубріння

Простое дерево без ограничений (переобучение)
Train accuracy = 63.43%
Train accuracy = 63.43%
Разрыв = -0.00%


# Cross-validator

In [46]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    RandomForestClassifier(n_estimators=200, max_depth=6, random_state = RANDOM_STATE),
    X, y, cv=5, scoring="accuracy"
)


ValueError: Found input variables with inconsistent numbers of samples: [1200, 12000000]

In [47]:
print(f"Accuracy на кажному з 5 розбиттів:\n{np.round(scores.mean():.2%)}")


SyntaxError: f-string: expecting '=', or '!', or ':', or '}' (3522117598.py, line 1)